<a href="https://colab.research.google.com/github/debashisdotchatterjee/-A-Generalized-P-lya-Urn-Model-with-Interaction-Dependent-Reinforcement/blob/main/51_Ion_Experimental_Bayesian_Horseshoe_Colab_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian inference from genuine 51-ion trapped-ion measurements

**Separate real-data extension** to *Sparse Quantum Hamiltonian Inference through Noncommuting Measurements*. Real measurements: [Zenodo DOI 10.5281/zenodo.17750419](https://doi.org/10.5281/zenodo.17750419); published experiment: [Phys. Rev. X 16, 031037 (2026)](https://doi.org/10.1103/s96t-n8tx).

The embedded experimental `.mat` file is from your previously uploaded research-output ZIP, not generated data. The **full published Hamiltonian** cannot be reconstructed from the publicly distributed randomized-unitary IDs without their corresponding rotation/basis metadata. We therefore fit an explicitly restricted nearest-neighbor closed-system quantum surrogate to the independent Z-basis Néel-state shots, with exact Born single-site marginals, fractional horseshoe inference and held-out comparisons. This limitation is recorded in the results.

Run all cells. Research mode will create and prompt downloading **`Quantum_51Ion_Real_Results.zip`**. The notebook works even if the optional Zenodo download is unavailable.

**October 29, 2026 corrigendum:** corrected a `uint8` overflow in the descriptive Neel-order summary; the original numerical curve was unphysically large and obscured nonlinear model predictions. Born-likelihood fitting and ridge/horseshoe algorithms are unchanged. The corrected code checks `abs(Neel order)<=1`.

In [1]:
# 1. Dependencies and execution options
!pip -q install "numpy>=1.24,<3" "scipy>=1.11" "pandas>=2.0" "matplotlib>=3.7" "tqdm>=4.65" "requests>=2.28"
PROFILE = "research"  # research | pilot | smoke. Use research for manuscript evidence.
RANDOM_SEED = 20261029
FRACTIONAL_POWER = 0.80
GLOBAL_HORSESHOE_SCALE = 0.16
print('Real-data profile:', PROFILE)

Real-data profile: research


In [2]:
# 2. Recover authentic experimental MAT file (embedded, byte-for-byte)
import base64, pathlib, hashlib
ROOT=pathlib.Path('/content/quantum_51ion_real')
(ROOT/'data/neel').mkdir(parents=True,exist_ok=True)
raw=base64.b64decode("""...
5EfpD7DagUWxI=""")
mat_path=ROOT/'data/neel/DataHL51IonmeasType14Base3.mat'
mat_path.write_bytes(raw)
print('Recovered experimentally recorded 51-ion Z-readout MAT:', len(raw),'bytes, SHA256',hashlib.sha256(raw).hexdigest())

Recovered experimentally recorded 51-ion Z-readout MAT: 73847 bytes, SHA256 f855babb1be76487f4e437f13c35946b4518ae5331f15c498c31d303003d1c3f


In [3]:
# 3. OPTIONAL randomized-measurement inventory (does not enable RU-based inference)
# The published archive omits the rotation/basis matrix for each RU ID. We only
# audit these files, and NEVER infer their unreported bases.
import requests, io, zipfile
u='https://zenodo.org/records/17750419/files/DataHLBicL51Ions.zip?download=1'
try:
    response=requests.get(u,timeout=60);response.raise_for_status()
    md5=hashlib.md5(response.content).hexdigest()
    if md5!='c130a911c71a297dbfbeb7305cd6ec36':
        raise ValueError('Unexpected MD5 for published randomized-measurement archive: '+md5)
    output=ROOT/'data/randomized';output.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(response.content)) as zi:
        for info in zi.infolist():
            if info.filename.endswith('.mat'):
                (output/pathlib.Path(info.filename).name).write_bytes(zi.read(info))
    print('Audited original published randomized file archive:',len(list(output.glob('*.mat'))),'MAT files. RU bases still unavailable.')
except Exception as e:
    print('Optional Zenodo randomized data download unavailable; actual Neel quantum fit is unaffected:',str(e)[:200])

Audited original published randomized file archive: 11 MAT files. RU bases still unavailable.


In [4]:
# 4. Write corrected, complete research-analysis source
# Fixes the uint8 overflow in the experimental Neel-order summary and plot.
from pathlib import Path
SOURCE = '#!/usr/bin/env python3\n"""51-ion real trapped-ion Z-basis dataset: constrained Bayesian Hamiltonian inference.\n\nDATA: Joshi et al., Zenodo 10.5281/zenodo.17750419. METHOD: Restricted\nnearest-neighbor free-fermion approximation, NOT the full Kraft et al. XX/YY\nlong-range + Lindblad inference. Exact 1-site Born likelihood from one site\nselected independently of the outcome for each independent full-register shot.\n"""\nimport argparse, sys, os, json, time, hashlib, math, zipfile, traceback, platform\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport numpy as np\nimport pandas as pd\nfrom scipy.io import loadmat\nfrom scipy.linalg import eigh_tridiagonal\nfrom scipy.optimize import minimize_scalar, minimize\nfrom scipy.special import expit, logit, gammaln\nfrom scipy.stats import beta as beta_dist\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nfrom tqdm.auto import tqdm\n\nDATA_DOI=\'10.5281/zenodo.17750419\'\nPAPER_DOI=\'10.1103/s96t-n8tx\'\nNION=51\nPROFILES={\'smoke\':dict(chains=2,burn=12,draws=15,thin=1,ridge_iter=14,grid=1),\n          \'pilot\':dict(chains=2,burn=100,draws=130,thin=2,ridge_iter=70,grid=3),\n          \'research\':dict(chains=2,burn=650,draws=650,thin=3,ridge_iter=180,grid=5)}\nSEED=20261029\n\nclass Steps:\n def __init__(self,root,names):\n  self.root=Path(root);self.status={name:\'pending\' for name in names};self.names=names;self.details={}\n def set(self,name,status,detail=\'\'):\n  self.status[name]=status;self.details[name]=detail;self.write()\n  print(f\'[{status.upper()}] {name}: {detail}\',flush=True)\n def write(self):\n  obj={\'tasks\':self.status,\'details\':self.details,\'completed\':sum(v==\'completed\' for v in self.status.values()),\'total\':len(self.status),\'updated_utc\':datetime.now(timezone.utc).isoformat()}\n  (self.root/\'progress.json\').write_text(json.dumps(obj,indent=2))\n  with open(self.root/\'PROGRESS.md\',\'w\') as f:\n   f.write(\'# Execution status (not scientific verification)\\n\\n\')\n   for k,v in self.status.items():f.write(f\'- **{v}** — {k}: {self.details.get(k,"")}\\n\')\n  fig,ax=plt.subplots(figsize=(9,2.1))\n  parts=[\'completed\',\'running\',\'skipped\',\'failed\',\'pending\'];palette={\'completed\':\'#117a65\',\'running\':\'#ca9133\',\'skipped\':\'#777b86\',\'failed\':\'#b83e42\',\'pending\':\'#d8dce2\'}\n  x=0\n  for status in parts:\n   n=sum(s==status for s in self.status.values());\n   if n:ax.barh([0],[n],left=x,height=.34,color=palette[status],label=f\'{status}: {n}\');x+=n\n  ax.set_xlim(0,len(self.names));ax.set_yticks([]);ax.set_xticks(range(len(self.names)+1));ax.set_xlabel(\'Pipeline stages\');ax.legend(loc=\'upper center\',bbox_to_anchor=(.5,1.33),ncol=5,frameon=False,fontsize=8)\n  ax.set_title(f\'{sum(s=="completed" for s in self.status.values())} / {len(self.names)} analysis stages completed\',pad=25,fontsize=12)\n  fig.savefig(self.root/\'PROGRESS.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\ndef style():\n plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.titlesize\':11,\'axes.labelsize\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'figure.dpi\':140,\'savefig.dpi\':220,\'pdf.fonttype\':42,\'axes.grid\':True,\'grid.alpha\':.15,\'legend.frameon\':False})\n\ndef savefig(root,stem):\n for ext in [\'png\',\'pdf\']:plt.savefig(Path(root)/\'figures\'/f\'{stem}.{ext}\',bbox_inches=\'tight\')\n plt.close()\n\ndef sha(path):\n h=hashlib.sha256()\n with open(path,\'rb\') as f:\n  for block in iter(lambda:f.read(1<<20),b\'\'):h.update(block)\n return h.hexdigest()\n\ndef load_neel(data_dir):\n matches=list(Path(data_dir).rglob(\'DataHL51IonmeasType14Base3.mat\'))\n if len(matches)!=1:raise FileNotFoundError(\'Need exactly one DataHL51IonmeasType14Base3.mat. Source archive in project/data/neel/.\')\n m=loadmat(matches[0],simplify_cells=True)[\'dataT\'];m=np.asarray(m,dtype=object).ravel()\n records=[]\n for batch,cell in enumerate(m):\n  cell=np.asarray(cell)\n  if cell.shape[1]!=51:raise ValueError(f\'Invalid matrix {cell.shape}; expected [times,1+50 shots]\')\n  for row in cell:\n   t=float(row[0]);vals=np.asarray(row[1:])\n   if not np.all(np.isfinite(vals)) or not np.all(vals==np.round(vals)):raise ValueError(\'Noninteger/invalid packed shot encountered\')\n   if np.any(vals<0) or np.any(vals>=2**51):raise ValueError(\'Some binary shot data exceed the 51-bit range\')\n   records.append((batch,t,vals.astype(np.uint64)))\n times=np.unique([r[1] for r in records]);out=[]\n for t in times:\n  arr=np.vstack([r[2] for r in records if r[1]==t]).ravel()\n  b=((arr[:,None]>>np.arange(51,dtype=np.uint64))&1).astype(np.uint8)\n  out.append(b)\n if len(times)!=11 or any(len(b)!=1000 for b in out):raise ValueError(\'Unexpected Neel sampling plan: expected 11 time points and 1000 shots/time\')\n if not np.array_equal(times,np.arange(0,11000,1000)):raise ValueError(f\'Unexpected time grid: {times}\')\n initial=(np.arange(51)%2==0).astype(np.uint8)\n rate=np.mean(out[0]!=initial[None,:])\n if rate>.10:raise ValueError(f\'t=0 readout mismatch {rate:.3f}; check bit orientation/state specification\')\n return times/1000.,out,initial,matches[0],rate\n\ndef inventory_randomized(data_dir):\n rows=[]\n for fn in sorted(Path(data_dir).rglob(\'DataHLBicL51Ions*.mat\')):\n  try:\n   cells=np.asarray(loadmat(fn,simplify_cells=True)[\'dataT\'],dtype=object).ravel()\n   ids=[];N=0\n   for c in cells:\n    c=np.asarray(c)\n    if c.ndim!=2 or c.shape[1]!=51:raise ValueError(\'wrong dimensions\')\n    ids.extend(c[:,0].astype(int).tolist());N+=c.shape[0]*(c.shape[1]-1)\n   rows.append({\'file\':fn.name,\'cells\':len(cells),\'unique_ru_ids\':len(set(ids)),\'shots\':N,\'status\':\'audited: basis rotations unavailable\'})\n  except Exception as e:rows.append({\'file\':fn.name,\'status\':f\'ERROR {e}\'})\n return pd.DataFrame(rows)\n\ndef split_counts(bits,seed=SEED):\n rng=np.random.default_rng(seed)\n K={s:np.zeros((len(bits)-1,51),dtype=int) for s in [\'train\',\'validation\',\'test\']}\n N={s:np.zeros((len(bits)-1,51),dtype=int) for s in K}\n assignments=[]\n for ti,B in enumerate(bits[1:]):\n  n=B.shape[0]; perm=rng.permutation(n); pos=int(round(.70*n));vpos=int(round(.80*n))\n  for name,idx in [(\'train\',perm[:pos]),(\'validation\',perm[pos:vpos]),(\'test\',perm[vpos:])]:\n   # Pre-determined site assignment is independent of the outcomes and is balanced.\n   sites=np.resize(rng.permutation(51),len(idx));rng.shuffle(sites)\n   obs=B[idx,sites]\n   N[name][ti]+=np.bincount(sites,minlength=51)\n   K[name][ti]+=np.bincount(sites,weights=obs,minlength=51).astype(int)\n   assignments.append({\'time_ms\':ti+1,\'split\':name,\'n_shots\':len(idx),\'n_sites\':len(np.unique(sites))})\n return K,N,pd.DataFrame(assignments)\n\nclass Fermion:\n """Exactly solvable 51-site nearest-neighbor spin-XY surrogate in Z basis.\n H=(1/2) sum_i J_i(X_i X_{i+1}+Y_i Y_{i+1}).\n In Jordan-Wigner representation, single-particle hopping is J_i.\n Initial occupations 1,0,1,...; n_j(t)=sum_{k initially occupied}|U_jk(t)|^2.\n The one-site Born likelihood is exact for this restricted quantum model.\n """\n def __init__(self,times_ms):self.times=np.asarray(times_ms)[1:];self.d=np.zeros(NION);self.init=np.arange(0,NION,2)\n def probs(self,logJ):\n  eig,Q=eigh_tridiagonal(self.d,np.exp(logJ),check_finite=False)\n  # Overlap with the occupied orbitals; Q real and orthogonal.\n  out=np.empty((len(self.times),NION))\n  Qt=Q.T[:,self.init]\n  for ti,t in enumerate(self.times):\n   A=(Q*np.exp(-1j*eig*t)[None,:])@Qt\n   out[ti]=np.sum(np.abs(A)**2,axis=1)\n  return np.clip(out,0,1)\n def observed(self,logJ,q):\n  p=self.probs(logJ)\n  return np.clip(q+(1-2*q)*p,1e-9,1-1e-9)\n\ndef loglike(K,N,prob):\n return float(np.sum(K*np.log(prob)+(N-K)*np.log1p(-prob)))\n\ndef nll(K,N,P):return -loglike(K,N,P)\n\ndef score(K,N,P):\n mask=N>0\n y=K[mask];n=N[mask];p=P[mask]\n return dict(n=int(n.sum()),mean_negative_logscore=float(-np.sum(y*np.log(p)+(n-y)*np.log1p(-p))/n.sum()),brier=float(np.sum(y*(1-p)**2+(n-y)*p**2)/n.sum()))\n\ndef theta_fit(model,K,N,q,pcfg):\n def fun(beta):return nll(K,N,model.observed(np.full(50,beta),q))\n # Long-time coherent evolution makes the likelihood multimodal in J; a\n # single bounded optimizer is NOT a reliable global search.\n grid=np.linspace(np.log(.035),np.log(3.0),140)\n vv=np.array([fun(g) for g in grid]); idx=np.argsort(vv)[:7];candidates=[]\n for j in idx:\n  l=max(0,j-1);u=min(len(grid)-1,j+1)\n  m=minimize_scalar(fun,bounds=(grid[l],grid[u]),method=\'bounded\',options={\'xatol\':1e-6})\n  if m.success:candidates.append((m.fun,m.x))\n if not candidates:raise RuntimeError(\'Homogeneous fit did not converge\')\n fun_min,beta_min=min(candidates)\n return float(beta_min),float(fun_min)\n\ndef ridge_fit(model,K,N,q,beta0,pcfg,lam,init=None):\n # penalized negative Binomial log likelihood with structured Tikhonov penalty.\n # Not the published Kraft et al. estimator; only an analogous regularization baseline.\n def obj(v):\n  bj=v[0]+v[1:]\n  if np.max(np.abs(bj))>5:return 1e7+np.sum(bj*bj)\n  loss=nll(K,N,model.observed(bj,q))\n  return loss+lam*(np.sum(v[1:]**2)+3*np.sum(np.diff(v[1:])**2)) +.5*((v[0]-np.log(.3))/.8)**2\n x=np.zeros(51) if init is None else np.asarray(init).copy()\n if init is None:x[0]=beta0\n bounds=[(np.log(.04),np.log(3.))]+[(-2,2)]*50\n r=minimize(obj,x,method=\'L-BFGS-B\',bounds=bounds,options={\'maxiter\':pcfg[\'ridge_iter\'],\'ftol\':2e-7,\'maxls\':15})\n return r\n\ndef logprior_lambdas(eta,tau,loglam):\n # conditional log density of u_j=log(lambda_j) given eta,tau, after Jacobian.\n # lambda_j half-Cauchy(0,1), eta_j|lambda_j,tau normal(0,(tau lambda_j)^2).\n return -.5*(eta/(tau*np.exp(loglam)))**2-np.logaddexp(0,2*loglam)\n\ndef logprior_tau(eta,loglam,u,scale):\n tau=np.exp(u);l=np.exp(loglam)\n return -len(eta)*u-.5*np.sum((eta/(tau*l))**2)+u-np.logaddexp(0,2*(u-np.log(scale)))\n\ndef slice_logscale(rng,x,logtarget,width=.45,max_steps=50):\n """Stepping-out univariate slice sampler on the real line (Neal 2003)."""\n level=logtarget(x)+np.log(rng.uniform()); left=x-rng.uniform()*width;right=left+width\n steps=rng.integers(0,max_steps+1);remaining=max_steps-steps\n while steps and logtarget(left)>level: left-=width; steps-=1\n while remaining and logtarget(right)>level:right+=width;remaining-=1\n for _ in range(250):\n  prop=rng.uniform(left,right)\n  if logtarget(prop)>=level:return prop\n  if prop<x:left=prop\n  else:right=prop\n raise RuntimeError(\'Slice sampler log-scale bracket did not close\')\n\ndef make_sampler(model,K,N,beta_init,q_init,n0_err,n0_total,pcfg,chain_seed,alpha=.8,tau_scale=.16):\n rng=np.random.default_rng(chain_seed)\n beta=float(beta_init+rng.normal(0,.025));eta=np.zeros(50)\n loglam=np.zeros(50);utau=np.log(.10);q=float(q_init)\n aq=1+n0_err;bq=1+n0_total-n0_err\n def ll(b,e,qq):return alpha*loglike(K,N,model.observed(b+e,qq))\n current=ll(beta,eta,q)\n acc={\'beta\':0,\'q\':0,\'lambda\':0,\'tau\':0,\'ess_calls\':0,\'ess_failures\':0}\n draws=[];niter=pcfg[\'burn\']+pcfg[\'draws\']*pcfg[\'thin\']\n for it in range(niter):\n  # Exact elliptical slice sampler for eta conditional on tau and local lambda.\n  sig=np.exp(utau+loglam)\n  nu=rng.normal(size=50)*sig\n  h=current+np.log(rng.uniform())\n  angle=rng.uniform(0,2*np.pi);lo=angle-2*np.pi;hi=angle\n  previous=eta.copy()\n  for attempt in range(120):\n   proposed=previous*np.cos(angle)+nu*np.sin(angle)\n   newll=ll(beta,proposed,q);acc[\'ess_calls\']+=1\n   if newll>h:\n    eta=proposed;current=newll;break\n   if angle<0:lo=angle\n   else:hi=angle\n   angle=rng.uniform(lo,hi)\n  else:\n   acc[\'ess_failures\']+=1\n   raise RuntimeError(\'Elliptical slice failed after 120 bracket reductions\')\n  # Metropolis updates for beta (global log hopping scale).\n  p=beta+rng.normal(0,.035)\n  nxt=ll(p,eta,q)\n  loga=nxt-current-.5*((p-np.log(.3))/.8)**2+.5*((beta-np.log(.3))/.8)**2\n  if np.log(rng.uniform())<loga:beta=p;current=nxt;acc[\'beta\']+=1\n  # q prior is calibrated by independent t=0 shots; update from t>0 as well.\n  uq=logit(q);qq=expit(uq+rng.normal(0,.055)); nxt=ll(beta,eta,qq)\n  # induced logit posterior log p(uq|..) is aq log q+bq log(1-q).\n  loga=nxt-current+aq*np.log(qq/q)+bq*np.log((1-qq)/(1-q))\n  if np.log(rng.uniform())<loga:q=qq;current=nxt;acc[\'q\']+=1\n  # Every local lambda has conditional Gibbs-with-MH update, cheaply done in log space.\n  tau=np.exp(utau)\n  for j in rng.permutation(50):\n   old=loglam[j];new=np.clip(old+rng.normal(0,.45),-12,12)\n   r=logprior_lambdas(eta[j],tau,old)\n   s=logprior_lambdas(eta[j],tau,new)\n   if np.log(rng.uniform())<(s-r):loglam[j]=new;acc[\'lambda\']+=1\n  utau=slice_logscale(rng,utau,lambda u:logprior_tau(eta,loglam,u,tau_scale),width=.5)\n  acc[\'tau\']+=1\n  if it>=pcfg[\'burn\'] and (it-pcfg[\'burn\'])%pcfg[\'thin\']==0:\n   draws.append(np.concatenate([[beta,q,np.exp(utau)],eta,np.exp(loglam)]))\n return np.asarray(draws),{k:(v/niter if k in [\'beta\',\'q\',\'tau\'] else (v/(niter*50) if k==\'lambda\' else v)) for k,v in acc.items()}\n\ndef gelman_rubin(arr):\n # arr: chains x samples x dims; classical split-R-hat with conservative finite-sample interpretation.\n c,n,d=arr.shape\n if n<4:return np.full(d,np.nan)\n half=n//2;x=np.concatenate((arr[:,:half,:],arr[:,half:2*half,:]),axis=0)\n W=x.var(axis=1,ddof=1).mean(axis=0); B=half*x.mean(axis=1).var(axis=0,ddof=1)\n return np.sqrt(((half-1)/half*W+B/half)/np.maximum(W,1e-20))\n\ndef chain_ess(y):\n # Initial-positive-sequence autocorrelation for 1D chains; approximation for diagnostics.\n a=np.asarray(y);c,n=a.shape\n if n<5:return np.nan\n a=a-a.mean(axis=1,keepdims=True);den=np.sum(a*a)/c/n\n if den<=1e-18:return np.nan\n rho=[]\n for lag in range(1,min(n//2,150)):\n  r=np.mean(np.sum(a[:,:-lag]*a[:,lag:],axis=1)/(n-lag))/den\n  rho.append(r)\n  if lag%2==0 and rho[-1]+rho[-2]<0:rho=rho[:-2];break\n return float(c*n/(1+2*sum(rho)))\n\ndef posterior_draws_to_preds(model,draws,max_draw=240,seed=SEED):\n rng=np.random.default_rng(seed);n=len(draws)\n idx=rng.choice(n,size=min(n,max_draw),replace=False)\n ps=[];js=[]\n for a in draws[idx]:\n  beta,q,tau=a[:3];eta=a[3:53];ps.append(model.observed(beta+eta,q));js.append(np.exp(beta+eta)*1000)\n return np.asarray(ps),np.asarray(js)\n\ndef png_figure_data(root,times,bits,init,rate):\n occupation=np.asarray([b.mean(axis=0) for b in bits]);z=1-2*occupation\n fig,ax=plt.subplots(1,2,figsize=(12,4.1))\n im=ax[0].imshow(z.T,origin=\'lower\',aspect=\'auto\',cmap=\'RdBu_r\',vmin=-1,vmax=1,extent=[times[0]-.5,times[-1]+.5,.5,51.5]);fig.colorbar(im,ax=ax[0],label=r\'$\\langle Z_j\\rangle$\')\n ax[0].set(xlabel=\'Evolution time (ms)\',ylabel=\'Ion index\',title=\'Measured single-spin polarization\')\n # IMPORTANT: init comes from packed shots as uint8. Convert BEFORE 2*init-1:\n # otherwise 2*uint8(0)-1 overflows to 255 and corrupts the Neel order.\n stagger=2.0*init.astype(np.float64)-1.0\n neel=stagger[None,:]*(2.0*occupation-1.0)\n order=neel.mean(axis=1)\n if not np.all(np.isfinite(order)) or np.any(np.abs(order)>1.0+1e-12):\n  raise AssertionError("Unphysical Neel order outside [-1,1]; inspect shot decoding or dtypes")\n ax[1].plot(times,order,\'o-\',label=\'Observed N\\u00e9el order\')\n ax[1].set(xlabel=\'Evolution time (ms)\',ylabel=\'Staggered occupation contrast\',title=\'Relaxation in 51-ion device\',ylim=(-.5,1.1));ax[1].legend()\n fig.tight_layout();savefig(root,\'01_experimental_neel_observables\')\n return occupation,order\n\ndef calibr_fig(root,rate,err,total):\n fig,ax=plt.subplots(figsize=(6.2,3.6));x=np.linspace(0,max(.045,rate*3),400);a=err+1;b=total-err+1\n ax.plot(x,beta_dist.pdf(x,a,b),color=\'#146c8b\',lw=2);ax.axvline(rate,color=\'#c27d30\',ls=\'--\',label=f\'Error proportion: {rate:.4f}\')\n ax.set(xlabel=\'Symmetric readout-flip probability\',ylabel=\'Calibration posterior density\',title=\'Calibration from known t=0 N\\u00e9el product state\');ax.legend()\n savefig(root,\'02_readout_calibration\')\n\ndef credible_figure(root,post,base,ridge):\n x=np.arange(1,51);lo=np.quantile(post,.05,axis=0);hi=np.quantile(post,.95,axis=0);md=np.median(post,axis=0)\n fig,ax=plt.subplots(figsize=(11.7,4.5))\n ax.fill_between(x,lo,hi,alpha=.19,color=\'#18739b\',label=\'Bayesian 90% marginal credible intervals\')\n ax.plot(x,md,lw=1.6,color=\'#135f83\',label=\'Horseshoe posterior median\')\n ax.plot(x,np.exp(ridge[0]+ridge[1:])*1000,color=\'#b36c2f\',alpha=.8,label=\'Tikhonov-style regularization\')\n ax.axhline(np.exp(base)*1000,color=\'gray\',ls=\'--\',label=\'Homogeneous coupling\')\n ax.set(xlabel=\'Nearest-neighbor bond index\',ylabel=r\'Effective $J_i$ (rad/s)\',title=\'Effective nearest-neighbor hopping under constrained XY model\');ax.legend(ncol=2)\n savefig(root,\'03_hopping_posterior\')\n\ndef shrinkage_fig(root,draws):\n eta=draws[:,3:53];ll=draws[:,53:103];tau=draws[:,2];fig,ax=plt.subplots(1,2,figsize=(11.5,4))\n ax[0].plot(np.arange(1,51),np.quantile(eta,[.05,.5,.95],axis=0).T,color=\'#146e89\',alpha=.55)\n ax[0].axhline(0,color=\'gray\',ls=\'--\');ax[0].set(xlabel=\'Bond index\',ylabel=\'Log-hopping deviation\',title=\'Horseshoe deviations: 5%, median, 95%\')\n ax[1].hist(tau,bins=32,density=True,alpha=.65,color=\'#b17435\');ax[1].set(xlabel=r\'Global horseshoe scale $\\tau$\',ylabel=\'Posterior density\',title=\'Global shrinkage uncertainty\')\n fig.tight_layout();savefig(root,\'04_global_local_shrinkage\')\n\ndef timeseries_fig(root,times,order,bits,init,PP,basem,ridgem):\n # Plot expectation of the global staggered order; raw shots have correlated sites.\n idx=np.where(init==1,1.,-1.)\n op=lambda p:np.sum((2*p-1)*idx[None,:],axis=1)/len(idx)\n experimental=order[1:];qt=np.quantile(np.asarray([op(p) for p in PP]),[.05,.5,.95],axis=0)\n fig,ax=plt.subplots(figsize=(9.3,4.5))\n ax.plot(times[1:],experimental,\'o\',label=\'Experiment (all Z bits)\',color=\'black\',ms=4)\n ax.plot(times[1:],qt[1],color=\'#146f96\',label=\'Horseshoe predictive median\')\n ax.fill_between(times[1:],qt[0],qt[2],color=\'#146f96\',alpha=.2,label=\'90% posterior parameter interval\')\n ax.plot(times[1:],op(basem),ls=\'--\',color=\'#808080\',label=\'Homogeneous\')\n ax.plot(times[1:],op(ridgem),ls=\':\',lw=2,color=\'#ad612c\',label=\'Tikhonov-style\')\n ax.set(xlabel=\'Evolution time (ms)\',ylabel=\'Staggered occupation contrast\',title=\'Observed N\\u00e9el order and restricted-model predictions\',ylim=(-1.05,1.05));ax.legend(ncol=2);savefig(root,\'05_neel_out_of_sample_trajectory\')\n\ndef residual_fig(root,bits,PP,times):\n observed=np.asarray([b.mean(0) for b in bits[1:]])\n predicted=PP.mean(0)\n fig,ax=plt.subplots(figsize=(11,4.4))\n im=ax.imshow((observed-predicted).T,origin=\'lower\',aspect=\'auto\',cmap=\'RdBu_r\',vmin=-.35,vmax=.35,extent=[.5,10.5,.5,51.5]);fig.colorbar(im,ax=ax,label=\'Observed − posterior predictive occupation\')\n ax.set(xlabel=\'Time (ms)\',ylabel=\'Ion index\',title=\'Physical model-discrepancy diagnostic: full-register summary\')\n savefig(root,\'06_residual_heatmap\')\n\ndef predictive_fig(root,K,N,pred,base,ridge):\n mets=[]\n for label,p in [(\'Horseshoe\',pred),(\'Tikhonov-style\',ridge),(\'Homogeneous\',base)]:\n  for i in range(10):\n   r=score(K[i:i+1],N[i:i+1],p[i:i+1]);mets.append(dict(method=label,time_ms=i+1,**r))\n df=pd.DataFrame(mets)\n fig,ax=plt.subplots(1,2,figsize=(11.8,4.2))\n for label,d in df.groupby(\'method\'):\n  for j,k in enumerate([\'mean_negative_logscore\',\'brier\']):ax[j].plot(d.time_ms,d[k],marker=\'o\',ms=3,label=label)\n for a,name in zip(ax,[\'Held-out negative log score\',\'Held-out Brier score\']):a.set(xlabel=\'Time (ms)\',ylabel=name,title=name+\' (lower is better)\');a.legend(fontsize=8)\n fig.tight_layout();savefig(root,\'07_test_scores_by_time\')\n return df\n\ndef site_calib_fig(root,K,N,P):\n x=P.ravel();pobs=np.divide(K,N,out=np.full_like(P,np.nan,dtype=float),where=N>0).ravel();weights=N.ravel();keep=(weights>0)\n df=pd.DataFrame({\'pred\':x[keep],\'obs\':pobs[keep],\'n\':weights[keep]})\n df[\'bin\']=pd.cut(df.pred,bins=np.linspace(0,1,11),include_lowest=True)\n grp=df.groupby(\'bin\',observed=True)[[\'pred\',\'obs\',\'n\']].apply(lambda a:pd.Series({\'predict\':np.average(a.pred,weights=a.n),\'observed\':np.average(a.obs,weights=a.n),\'shots\':a.n.sum()})).dropna()\n fig,ax=plt.subplots(figsize=(5.8,5.1));ax.plot([0,1],[0,1],ls=\'--\',c=\'black\',lw=1)\n ax.scatter(grp.predict,grp.observed,s=np.sqrt(grp.shots)*4,color=\'#18729a\')\n ax.set(xlabel=\'Predicted single-site probability\',ylabel=\'Observed held-out frequency\',xlim=(0,1),ylim=(0,1),title=\'Independent-shot calibration\');savefig(root,\'08_holdout_calibration\')\n return grp\n\ndef diagnostics_fig(root,chains):\n fig,ax=plt.subplots(2,2,figsize=(11.5,6.8));title=[\'Global log coupling\',\'Readout flip probability\',\'Horseshoe global tau\',\'Bond 25 log-deviation\'];inds=[0,1,2,3+24]\n for a,ind,ttl in zip(ax.ravel(),inds,title):\n  for k,c in enumerate(chains):a.plot(c[:,ind],alpha=.65,label=f\'Chain {k+1}\',lw=.7)\n  a.set_title(ttl);a.set_xlabel(\'Retained draw\');a.legend(fontsize=8)\n fig.tight_layout();savefig(root,\'09_mcmc_traces\')\n\ndef save_table(root,df,name,fmt=\'%.5g\'):\n dest=Path(root)/\'tables\';df.to_csv(dest/f\'{name}.csv\',index=False)\n with open(dest/f\'{name}.tex\',\'w\') as f:\n  f.write(\'% Auto-generated numerical LaTeX table; review column sizes before manuscript inclusion.\\n\')\n  f.write(df.to_latex(index=False,float_format=lambda z:fmt%z if np.isfinite(z) else \'NA\',escape=True))\n\ndef paired_shot_loss_differences(K,N,Pi,Pj,seed=SEED):\n """Exactly reconstruct paired per-shot log-loss differences from binomial counts.\n Pointwise comparison uses identical selected test shots; time-block bootstrap\n acknowledges potential shared time-specific drift, but only 10 time blocks.\n """\n dif=[];sums=[];ns=[]\n for t in range(len(K)):\n  dpos=np.log(np.clip(Pj[t],1e-12,1))-np.log(np.clip(Pi[t],1e-12,1))\n  dneg=np.log1p(-Pj[t])-np.log1p(-Pi[t])\n  dt=np.concatenate((np.repeat(dpos,K[t]),np.repeat(dneg,N[t]-K[t])))\n  dif.append(dt);sums.append(np.sum(dt));ns.append(len(dt))\n d=np.concatenate(dif);rng=np.random.default_rng(seed)\n boots=[]\n for i in range(5000):\n  ix=rng.integers(0,len(sums),size=len(sums))\n  boots.append(np.sum(np.asarray(sums)[ix])/np.sum(np.asarray(ns)[ix]))\n return {\'difference_negative_logscore\':float(d.mean()),\'shotlevel_standard_error\':float(d.std(ddof=1)/np.sqrt(len(d))),\'shotlevel_CI95_low\':float(d.mean()-1.96*d.std(ddof=1)/np.sqrt(len(d))),\'shotlevel_CI95_high\':float(d.mean()+1.96*d.std(ddof=1)/np.sqrt(len(d))),\'timeblock_bootstrap_CI95_low\':float(np.quantile(boots,.025)),\'timeblock_bootstrap_CI95_high\':float(np.quantile(boots,.975)),\'times\':len(sums),\'shots\':len(d)}\n\ndef plot_method_comparison(root,df):\n fig,ax=plt.subplots(figsize=(8,4));col=[\'#176d8c\',\'#bd793b\',\'#777777\']\n a=df.set_index(\'method\');names=[\'Horseshoe\',\'Tikhonov-style\',\'Homogeneous\'];x=np.arange(3)\n ax.bar(x,[a.loc[n,\'test_mean_negative_logscore\'] for n in names],color=col)\n ax.set_xticks(x,names);ax.set_ylabel(\'Held-out negative log score (nats/shot)\');ax.set_title(\'Independent test shots; identical measurement selection\')\n for i,n in enumerate(names):ax.text(i,a.loc[n,\'test_mean_negative_logscore\']+.002,f"{a.loc[n,\'test_mean_negative_logscore\']:.4f}",ha=\'center\',fontsize=9)\n savefig(root,\'10_model_comparison\')\n\ndef gauge_fig(root,model,beta):\n # Z-gauge: diagonal phase rotations reverse any individual hopping sign with no\n # consequence for occupation measurements from a Z-diagonal initial state.\n # This is a fundamental limitation, not a failed optimization.\n eig,Q=eigh_tridiagonal(np.zeros(51),np.exp(np.full(50,beta)))\n p=model.probs(np.full(50,beta)); \n # Gauge transform D_{j}=prod_{k<j} sign(J_k) for arbitrary edge sign vector.\n sign=np.ones(50);sign[8]=-1;sign[19]=-1;sign[33]=-1\n d=np.cumprod(np.r_[1.,sign]);\n # U\'=D U D, so |U\'_{jk}|^2=|U_{jk}|^2 exactly.\n # Demonstrate numerically with signed tridiagonal eigendecomposition.\n E,Q2=eigh_tridiagonal(np.zeros(51),np.exp(np.full(50,beta))*sign)\n pp=[]\n for t in model.times:\n  U=(Q2*np.exp(-1j*E*t)[None,:])@Q2.T\n  pp.append(np.sum(np.abs(U[:,::2])**2,axis=1))\n maxerr=float(np.max(np.abs(p-np.asarray(pp))))\n fig,ax=plt.subplots(figsize=(8,3.6));ax.plot(model.times,np.max(np.abs(p-np.asarray(pp)),axis=1),\'o-\',color=\'#386e93\')\n ax.set(xlabel=\'Evolution time (ms)\',ylabel=\'Maximum site-probability difference\',title=\'Exact Z-basis sign-gauge non-identifiability\')\n savefig(root,\'11_sign_gauge_identifiability\')\n return maxerr\n\ndef main(args):\n style();root=Path(args.out).resolve();(root/\'figures\').mkdir(parents=True,exist_ok=True);(root/\'tables\').mkdir(exist_ok=True)\n names=[\'Dataset reconstruction and audit\',\'Independent-shot split and readout calibration\',\'Homogeneous and Tikhonov baselines\',\'Fractional horseshoe posterior\',\'Posterior convergence and identifiability\',\'Out-of-sample predictions and figures\',\'Early-time-to-late-time extrapolation\',\'Published-method scope audit\',\'Archive and reproducibility manifest\'];prog=Steps(root,names);prog.write()\n start=time.time();pcfg=PROFILES[args.profile]\n try:\n  times,bits,init,mat,rate=load_neel(args.data_dir)\n  audit=inventory_randomized(args.data_dir);save_table(root,audit,\'randomized_hamiltonian_archive_inventory\')\n  occ,order=png_figure_data(root,times,bits,init,rate)\n  n0_total=int(bits[0].size);n0_err=int(np.sum(bits[0]!=init[None,:]));q0=(n0_err+1)/(n0_total+2)\n  save_table(root,pd.DataFrame([{\'time_ms\':times[i],\'shots\':len(bits[i]),\'spin_up_fraction_mean\':1-occ[i].mean(),\'staggered_order\':order[i]} for i in range(len(times))]),\'real_data_observables\')\n  prog.set(names[0],\'completed\',f\'11 times, 1000 independently recorded 51-spin shots/time; t0 mismatch={rate:.4f}; randomized RU basis metadata missing.\')\n  K,N,split_df=split_counts(bits,args.seed);save_table(root,split_df,\'split_inventory\');calibr_fig(root,q0,n0_err,n0_total)\n  (root/\'data_provenance.json\').write_text(json.dumps({\'dataset_doi\':DATA_DOI,\'published_article_doi\':PAPER_DOI,\'input_sha256\':sha(mat),\'raw_files_source\':\'Source Zenodo, repacked as MAT from user-uploaded research simulation results ZIP.\',\'raw_shots\':int(sum(len(x) for x in bits)),\'spin_readout_bits\':int(sum(x.size for x in bits)),\'readout_calibration_error_n\':n0_err,\'readout_calibration_total_n\':n0_total,\'readout_calibration_posterior\':f\'Beta({n0_err+1},{n0_total-n0_err+1})\',\'randomized_metadata_warning\':\'RU basis and preparation rotations absent from public deposit; no physical likelihood for randomized archive constructed.\'},indent=2))\n  prog.set(names[1],\'completed\',\'Random 70% train / 10% validation / 20% test shot splits; one outcome-independent ion per shot; t=0 calibration independent of time>0 fit.\')\n  model=Fermion(times);best,_=theta_fit(model,K[\'train\'],N[\'train\'],q0,pcfg)\n  lams=np.logspace(-.7,2,pcfg[\'grid\']);cand=[]\n  for lam in tqdm(lams,desc=\'Tikhonov validation grid\'):\n   r=ridge_fit(model,K[\'train\'],N[\'train\'],q0,best,pcfg,lam)\n   val=score(K[\'validation\'],N[\'validation\'],model.observed(r.x[0]+r.x[1:],q0))[\'mean_negative_logscore\']\n   cand.append({\'ridge_penalty\':float(lam),\'validation_negative_logscore\':val,\'optimization_success\':bool(r.success),\'optimizer_message\':str(r.message),\'iterations\':int(r.nit)})\n  cand_df=pd.DataFrame(cand);save_table(root,cand_df,\'ridge_validation\')\n  chosen=float(cand_df.sort_values(\'validation_negative_logscore\').iloc[0].ridge_penalty)\n  Kfit=K[\'train\']+K[\'validation\'];Nfit=N[\'train\']+N[\'validation\']\n  bfit,_=theta_fit(model,Kfit,Nfit,q0,pcfg)\n  rr=ridge_fit(model,Kfit,Nfit,q0,bfit,pcfg,chosen)\n  if not np.isfinite(rr.fun):raise RuntimeError(\'Ridge optimizer returned nonfinite value\')\n  prog.set(names[2],\'completed\',f\'Uniform J={1000*np.exp(bfit):.3f} rad/s; selected Tikhonov-style lambda={chosen:.4g}.\')\n  alpha=args.alpha;chains=[];accept=[]\n  for ch in tqdm(range(pcfg[\'chains\']),desc=\'Independent horseshoe chains\'):\n   draws,dia=make_sampler(model,Kfit,Nfit,bfit,q0,n0_err,n0_total,pcfg,args.seed+101+ch*107,alpha=alpha,tau_scale=args.tau_scale)\n   chains.append(draws);accept.append({\'chain\':ch+1,**dia})\n  arr=np.stack(chains);all_draws=arr.reshape(-1,arr.shape[-1]);rhat=gelman_rubin(arr)\n  save_table(root,pd.DataFrame(accept),\'mcmc_acceptance\')\n  pd.DataFrame(all_draws,columns=[\'log_J_global_rad_per_ms\',\'readout_flip_q\',\'tau\']+[f\'log_J_deviation_bond_{i+1:02d}\' for i in range(50)]+[f\'local_lambda_bond_{i+1:02d}\' for i in range(50)]).to_csv(root/\'tables\'/\'horseshoe_posterior_draws.csv\',index=False)\n  save_table(root,pd.DataFrame({\'parameter\':[\'global_log_J\',\'q\',\'tau\',\'bond25_log_dev\'],\'rhat\':[rhat[0],rhat[1],rhat[2],rhat[3+24]],\'ess\':[chain_ess(arr[:,:,i]) for i in [0,1,2,3+24]]}),\'mcmc_diagnostics\')\n  save_table(root,pd.DataFrame({\'parameter\':[\'beta\',\'q\',\'tau\']+[f\'eta_{i+1}\' for i in range(50)],\'rhat\':rhat[:53]}),\'all_53_parameter_rhat\')\n  prog.set(names[3],\'completed\',f\'{len(all_draws)} retained posterior draws, alpha={alpha}; valid independently selected one-site Born likelihood for restricted 51-ion XY model.\')\n  PP,JJ=posterior_draws_to_preds(model,all_draws,max_draw=args.predictive_draws,seed=args.seed)\n  p_b=model.observed(np.full(50,bfit),q0);p_r=model.observed(rr.x[0]+rr.x[1:],q0);p_h=np.mean(PP,axis=0)\n  gauge_error=gauge_fig(root,model,bfit)\n  diagnostics_fig(root,chains);credible_figure(root,JJ,bfit,rr.x);shrinkage_fig(root,all_draws)\n  if gauge_error>1e-10:raise AssertionError(\'Gauge invariance numerical test failed\')\n  convergence_note=(\'Rhat warning: further chains/longer sampling recommended\' if np.nanmax(rhat[:53])>1.1 else \'Rhat at or below 1.1 on recorded parameters\')\n  prog.set(names[4],\'completed\',f\'Gauge invariance max discrepancy {gauge_error:.2e}; MCMC Rhat max {np.nanmax(rhat[:53]):.3f}. {convergence_note}.\')\n  modelrows=[]\n  for method,P in [(\'Horseshoe\',p_h),(\'Tikhonov-style\',p_r),(\'Homogeneous\',p_b)]:\n   tr=score(Kfit,Nfit,P);te=score(K[\'test\'],N[\'test\'],P)\n   modelrows.append({\'method\':method,\'train_mean_negative_logscore\':tr[\'mean_negative_logscore\'],\'test_mean_negative_logscore\':te[\'mean_negative_logscore\'],\'test_brier\':te[\'brier\'],\'test_shots\':te[\'n\']})\n  model_df=pd.DataFrame(modelrows);save_table(root,model_df,\'heldout_model_comparison\')\n  paired=[]\n  for competitor,P in [(\'Tikhonov-style\',p_r),(\'Homogeneous\',p_b)]:\n   paired.append({\'comparison\':f\'Horseshoe minus {competitor}\',**paired_shot_loss_differences(K[\'test\'],N[\'test\'],p_h,P,seed=args.seed)})\n  paired_df=pd.DataFrame(paired);save_table(root,paired_df,\'paired_predictive_difference_uncertainty\')\n  fig,ax=plt.subplots(figsize=(8.6,3.7));x=np.arange(len(paired_df))\n  ax.axhline(0,color=\'black\',lw=.8,ls=\'--\')\n  ax.errorbar(x,paired_df.difference_negative_logscore,yerr=np.vstack([paired_df.difference_negative_logscore-paired_df.timeblock_bootstrap_CI95_low,paired_df.timeblock_bootstrap_CI95_high-paired_df.difference_negative_logscore]),fmt=\'o\',capsize=4,color=\'#196e8c\')\n  ax.set_xticks(x,paired_df.comparison,rotation=9);ax.set(ylabel=\'Paired held-out loss difference (nats/shot)\',title=\'Predictive comparisons with time-block bootstrap 95% intervals\');savefig(root,\'13_comparison_uncertainty\')\n  tdf=predictive_fig(root,K[\'test\'],N[\'test\'],p_h,p_b,p_r);save_table(root,tdf,\'test_scores_by_time\')\n  timeseries_fig(root,times,order,bits,init,PP,p_b,p_r);residual_fig(root,bits,PP,times)\n  calib=site_calib_fig(root,K[\'test\'],N[\'test\'],p_h);save_table(root,calib.reset_index().astype({\'bin\':str}),\'heldout_calibration\')\n  jrec=[];m=np.median(JJ,axis=0);lo,hi=np.quantile(JJ,[.05,.95],axis=0)\n  for i in range(50):jrec.append({\'bond\':i+1,\'J_med_rad_per_s\':m[i],\'J_90low\':lo[i],\'J_90high\':hi[i],\'tikhonov_J_rad_per_s\':1000*np.exp(rr.x[0]+rr.x[i+1]),\'homogeneous_J_rad_per_s\':1000*np.exp(bfit)})\n  save_table(root,pd.DataFrame(jrec),\'bondwise_posterior_couplings\')\n  meta={\'alpha\':alpha,\'tau_scale\':args.tau_scale,\'profile\':args.profile,\'seed\':args.seed,\'MCMC_chains\':pcfg[\'chains\'],\'burnin\':pcfg[\'burn\'],\'draws_per_chain\':pcfg[\'draws\'],\'thin\':pcfg[\'thin\'],\'heldout_summary\':model_df.to_dict(orient=\'records\'),\'ridge_convergence\':bool(rr.success),\'ridge_optimization_message\':rr.message,\'gauge_max_error\':gauge_error,\'max_Rhat_first_53\':float(np.max(rhat[:53])),\'model_scope\':\'Nearest-neighbor XX+YY, fixed pure Neel initial state, symmetric readout flip, no Lindbladian; model misspecification possible.\'}\n  (root/\'numerical_summary.json\').write_text(json.dumps(meta,indent=2,default=str))\n  prog.set(names[5],\'completed\',f\'Independent 2000 test shots; {len(list((root/"figures").glob("*.png")))} figures; performance measured without ground truth Hamiltonian.\')\n  if args.profile==\'research\':\n   # This is a SEPARATE refit using early-time data only. No leakage from\n   # late-time measurements is allowed into these early-trained estimators.\n   early=5\n   Ke=Nfit*0; Ne=Nfit*0\n   Ke[:early]=Kfit[:early];Ne[:early]=Nfit[:early]\n   b_early,_=theta_fit(model,Ke,Ne,q0,pcfg)\n   # Early validation used for tuning; refit on early train + validation only.\n   early_grid=[]\n   Ketr=K[\'train\']*0;Netr=N[\'train\']*0\n   Ketr[:early]=K[\'train\'][:early];Netr[:early]=N[\'train\'][:early]\n   btr,_=theta_fit(model,Ketr,Netr,q0,pcfg)\n   for lam in np.logspace(-.7,2,pcfg[\'grid\']):\n    rv=ridge_fit(model,Ketr,Netr,q0,btr,pcfg,lam)\n    Pv=model.observed(rv.x[0]+rv.x[1:],q0)\n    val=score(K[\'validation\'][:early],N[\'validation\'][:early],Pv[:early])[\'mean_negative_logscore\']\n    early_grid.append((val,lam))\n   lam_e=min(early_grid)[1]\n   re=ridge_fit(model,Ke,Ne,q0,b_early,pcfg,lam_e)\n   small=dict(chains=2,burn=360,draws=360,thin=2,ridge_iter=pcfg[\'ridge_iter\'],grid=3)\n   temporal=[]\n   for ch in range(2):\n    dc,_=make_sampler(model,Ke,Ne,b_early,q0,n0_err,n0_total,small,args.seed+901+ch*151,alpha=args.alpha,tau_scale=args.tau_scale)\n    temporal.append(dc)\n   temporal=np.stack(temporal)\n   PEt,_=posterior_draws_to_preds(model,temporal.reshape(-1,temporal.shape[-1]),max_draw=200,seed=args.seed+17)\n   pp=PEt.mean(axis=0)\n   evalK=K[\'train\']+K[\'validation\']+K[\'test\'];evalN=N[\'train\']+N[\'validation\']+N[\'test\']\n   extrap=[]\n   for name,P in [(\'Horseshoe\',pp),(\'Tikhonov-style\',model.observed(re.x[0]+re.x[1:],q0)),(\'Homogeneous\',model.observed(np.full(50,b_early),q0))]:\n    for i in range(early,len(model.times)):\n     vals=score(evalK[i:i+1],evalN[i:i+1],P[i:i+1])\n     extrap.append({\'method\':name,\'time_ms\':float(model.times[i]),\'NLL_per_shot\':vals[\'mean_negative_logscore\'],\'Brier\':vals[\'brier\'],\'n_independent_shots\':vals[\'n\']})\n   te=pd.DataFrame(extrap);save_table(root,te,\'temporal_extrapolation\')\n   fig,ax=plt.subplots(figsize=(8.7,4.1))\n   for method,d in te.groupby(\'method\'):\n    ax.plot(d.time_ms,d.NLL_per_shot,\'o-\',label=method)\n   ax.set(xlabel=\'Held-out evolution time (ms)\',ylabel=\'Negative log score (nats/shot)\',title=\'Early-time training (1–5 ms); forward prediction (6–10 ms)\');ax.legend();savefig(root,\'12_temporal_extrapolation\')\n   trhat=gelman_rubin(temporal)\n   save_table(root,pd.DataFrame({\'parameter\':[\'global_log_J\',\'q\',\'tau\'],\'earlyfit_rhat\':trhat[:3]}),\'temporal_mcmc_diagnostics\')\n   prog.set(names[6],\'completed\',f\'Independent refits restricted to 1–5 ms; forecast measured at 6–10 ms; early-fit tau Rhat={trhat[2]:.3f}.\')\n  else:\n   prog.set(names[6],\'skipped\',\'Temporal refit reserved for research profile; all standard held-out analyses still run.\')\n\n  scope=\'\'\'# What may and may not be claimed from this dataset\\n\\n\'\'\'+\'\'\'The public 51-ion randomized-unitary archive records numeric RU IDs but not their actual 51-qubit preparation and measurement rotations. It is impossible to evaluate the exact full Hamiltonian Born likelihood from the published .mat files alone. The separately deposited Neel dataset has a documented product state and Z measurements and supports the analysis here.\\n\\nThis package models the physically constrained nearest-neighbor H=(1/2)sum J_j(X_jX_{j+1}+Y_jY_{j+1}); Jordan-Wigner gives exact free-fermion single-site probabilities for Z-readout. Signs of hopping are not identifiable: only magnitudes are inferred, a fundamental gauge symmetry. The 51-ion experiment is a long-range interacting OPEN system. The restricted closed-system model is an effective, potentially misspecified surrogate, not an estimate of the full physical generator.\\n\\nThe one-ion-per-shot selection makes the Bernoulli likelihood a valid ordinary likelihood under independent shots, instead of falsely assuming that all 51 outcomes within a shot are independent. The fractional power applies only to this likelihood; a separate t=0 calibration likelihood updates a common readout parameter. Time-split validation for distant times is not obtained by a random shot holdout across all times; observed-vs-predicted later dynamics is descriptive.\\n\\nKraft et al., Physical Review X 16 031037 (2026), DOI 10.1103/s96t-n8tx, learn a long-range coherent+dissipative generator using randomized local measurements, pair and triple correlators and inverse-problem regression with Tikhonov regularization. Their paper reports nominal 51-ion coupling error-bar sizes changing from around 40 rad/s to around 18 rad/s after regularization. Those numbers are only contextual and NOT directly comparable with our credible interval widths, model predictions or our independent-shot Tikhonov-style surrogate. We provide a same-Z-data comparison against a Tikhonov-penalized effective-model likelihood, NOT a claim of reproducing the PRX published estimator. Do not claim direct agreement or superiority over their full H+L model without RU basis metadata and the published output parameters.\\n\\nRecommended next step: request randomized measurement bases and state-preparation rotations by RU ID from the dataset authors, along with published parameter estimates and covariance matrices. Then reconstruct the full operator-learning design and perform an apples-to-apples comparison.\\n\'\'\'\n  (root/\'COMPARISON_SCOPE_AND_LIMITATIONS.md\').write_text(scope)\n  save_table(root,pd.DataFrame([{\'method\':\'Kraft et al. published H+L learning\',\'data\':\'Randomized local preparations and measurements\',\'physical_model\':\'Long-range Ising/XX-YY plus Lindbladian\',\'comparison\':\'Publication context ONLY; no full reconstruction\'}, {\'method\':\'Tikhonov-style baseline here\',\'data\':\'Independent Z-shot outcome, one ion/shot\',\'physical_model\':\'Nearest-neighbor closed XY surrogate\',\'comparison\':\'Direct Z-data predictive comparison\'}, {\'method\':\'Fractional horseshoe here\',\'data\':\'Same independent Z-shot outcomes\',\'physical_model\':\'Same nearest-neighbor closed XY surrogate\',\'comparison\':\'Direct Z-data predictive comparison\'}]),\'published_method_scope\')\n  prog.set(names[7],\'completed\',\'Same-data Tikhonov baseline computed; published H+L method accurately contextualized but not falsely represented as reproduced.\')\n  status={\'DATA\':f\'{len(bits)} times, {len(bits[0])} shots/time, 51 ions\',\'INFERENCE\':f\'horseshoe with {len(all_draws)} retained samples\', \'FIT\':model_df.to_string(index=False),\'WARNING\':\'This is restricted nearest-neighbor XY fitting; published 51-ion Hamiltonian has long-range couplings/dissipation.\'}\n  with open(root/\'REPORT_READ_FIRST.md\',\'w\') as f:\n   f.write(\'# 51-ion real-data Bayesian analysis: execution report\\n\\n\')\n   for k,v in status.items():f.write(f\'## {k}\\n\\n{v}\\n\\n\')\n   f.write(\'## Provenance and claims\\n\\nSee data_provenance.json and COMPARISON_SCOPE_AND_LIMITATIONS.md. All plots and tables correspond to actual experimental Z-basis measurements. No 51-ion full Hamiltonian reconstruction is claimed.\\n\')\n  manifest={\'generated_utc\':datetime.now(timezone.utc).isoformat(),\'python\':sys.version,\'numpy\':np.__version__,\'scipy\':__import__(\'scipy\').__version__,\'datasets\':{\'zenodo_doi\':DATA_DOI,\'paper_doi\':PAPER_DOI},\'files_sha256\':{}}\n  for fn in sorted(root.rglob(\'*\')):\n   if fn.is_file() and fn.name!=\'manifest.json\':manifest[\'files_sha256\'][str(fn.relative_to(root))]=sha(fn)\n  (root/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n  prog.set(names[8],\'completed\',\'CSV, TeX, figures, model-draws, provenance, methodological limitations and manifest produced.\')\n except Exception as e:\n  current=next((s for s,v in prog.status.items() if v==\'pending\'),None)\n  if current:prog.set(current,\'failed\',str(e))\n  (root/\'error.log\').write_text(traceback.format_exc())\n  print(traceback.format_exc(),flush=True)\n finally:\n  (root/\'runtime_seconds.txt\').write_text(str(time.time()-start))\n  zipname=root.parent/(root.name+\'.zip\')\n  with zipfile.ZipFile(zipname,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as out:\n   for p in root.rglob(\'*\'):\n    if p.is_file():out.write(p,arcname=str(Path(root.name)/p.relative_to(root)))\n  print(f\'\\nRESULTS ZIP: {zipname}\\n\',flush=True)\n\nif __name__==\'__main__\':\n ap=argparse.ArgumentParser()\n ap.add_argument(\'--profile\',choices=PROFILES,default=\'research\')\n ap.add_argument(\'--data-dir\',default=str(Path(__file__).parent/\'data\'))\n ap.add_argument(\'--out\',default=\'Quantum_51Ion_Real_Results\')\n ap.add_argument(\'--seed\',type=int,default=SEED)\n ap.add_argument(\'--alpha\',type=float,default=.8)\n ap.add_argument(\'--tau-scale\',type=float,default=.16)\n ap.add_argument(\'--predictive-draws\',type=int,default=300)\n main(ap.parse_args())\n'
script_path=ROOT/'real_51ion_analysis.py'
script_path.write_text(SOURCE)
print('Corrected research source installed:',script_path,'size:',len(SOURCE))

Corrected research source installed: /content/quantum_51ion_real/real_51ion_analysis.py size: 40409


In [5]:
# 5. Execute REAL-DATA inference and generate tables/plots/status/results ZIP
import subprocess, sys, os
cmd=[sys.executable,str(script_path),'--profile',PROFILE,'--data-dir',str(ROOT/'data'),'--out','/content/Quantum_51Ion_Real_Results','--seed',str(RANDOM_SEED),'--alpha',str(FRACTIONAL_POWER),'--tau-scale',str(GLOBAL_HORSESHOE_SCALE)]
env=os.environ.copy();env.update({'OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1','OMP_NUM_THREADS':'1'})
print('Running:', ' '.join(cmd))
result=subprocess.run(cmd,env=env,check=False)
if result.returncode:
    raise RuntimeError(f'Analysis failed (code {result.returncode}); open /content/Quantum_51Ion_Real_Results/error.log for details. A partial diagnostics ZIP is still generated.')
print('COMPLETE: all requested stages attempted; check PROGRESS.md and MCMC diagnostics for warnings.')

Running: /usr/bin/python3 /content/quantum_51ion_real/real_51ion_analysis.py --profile research --data-dir /content/quantum_51ion_real/data --out /content/Quantum_51Ion_Real_Results --seed 20261029 --alpha 0.8 --tau-scale 0.16
COMPLETE: all requested stages attempted; check PROGRESS.md and MCMC diagnostics for warnings.


In [6]:
# 6. Confirm and download the generated results ZIP
from google.colab import files
output=Path('/content/Quantum_51Ion_Real_Results.zip')
assert output.exists(), 'No ZIP found; inspect the preceding cell for an error.'
print('Final ZIP size:',round(output.stat().st_size/1e6,2),'MB')
files.download(str(output))

Final ZIP size: 3.47 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Interpretation:** The package includes the real recorded outcomes, posterior marginal coupling maps, independent-shot held-out log score and Brier score, comparative bootstrap CIs, sign-gauge check, MCMC diagnostics and (in `research` mode) out-of-time forecast. It **cannot** establish the entire 51-ion long-range Lindbladian or a direct numerical comparison with Kraft et al.'s published fit, because essential RU basis metadata are absent. Please upload the ZIP for a formal manuscript-level discussion.

**Figure sanity check:** the experimental staggered occupation contrast in Fig. 05 and in `real_data_observables.csv` must lie in [-1,1]. If your earlier run showed values such as -90 or +40, it was affected by the former unsigned-integer arithmetic bug; do not use that figure/table.